# Parcial 4 — MM3014 Teoría de Probabilidades
## Simulaciones Monte Carlo

| Parámetro | Valor |
|-----------|-------|
| Semilla global | `np.random.seed(2026)` |
| Repeticiones | R = 10 000 |
| Decimales de salida | 4 |

In [ ]:
import numpy as np

SEED = 2026
R    = 10_000

---
## Problema A – Dados

**Experimento base:** Simular el lanzamiento de dos dados justos de 6 caras.

- **a.** Estimar la probabilidad de que la suma sea igual a 7.
- **b.** Estimar la probabilidad de que la suma sea 7 dado que al menos uno de los dados es par.

In [ ]:
np.random.seed(SEED)

# Simular R lanzamientos de dos dados (valores 1-6)
dado1 = np.random.randint(1, 7, size=R)
dado2 = np.random.randint(1, 7, size=R)
suma  = dado1 + dado2

# a. P(suma = 7)
p_suma_7 = np.mean(suma == 7)

# b. P(suma = 7 | al menos uno de los dados es par)
al_menos_un_par = (dado1 % 2 == 0) | (dado2 % 2 == 0)
p_cond = np.sum((suma == 7) & al_menos_un_par) / np.sum(al_menos_un_par)

print(f"P(suma = 7) = {p_suma_7:.4f}")
print(f"P(suma = 7 | al menos un par) = {p_cond:.4f}")

---
## Problema B – Monedas

**Experimento base:** Simular el lanzamiento de tres monedas justas (`0` = cruz, `1` = cara).

- **a.** Estimar la probabilidad de obtener exactamente dos caras.
- **b.** Sea $X$ el número de caras. Estimar $E[X]$.

In [ ]:
np.random.seed(SEED)

# Simular R experimentos de 3 monedas; 0=cruz, 1=cara
lanzamientos = np.random.randint(0, 2, size=(R, 3))
caras = lanzamientos.sum(axis=1)  # nro. de caras por experimento

# a. P(exactamente 2 caras)
p_2_caras = np.mean(caras == 2)

# b. E[X], donde X = número de caras
e_x = np.mean(caras)

print(f"P(exactamente 2 caras) = {p_2_caras:.4f}")
print(f"E[X] = {e_x:.4f}")

---
## Problema C – Canicas de colores

**Codificación:** `0` = roja | `1` = azul | `2` = verde

**Parte 1:** Una caja con 5 rojas, 3 azules y 2 verdes. Extraer 2 **sin reemplazo**. Estimar $P(\text{ambas rojas})$.

**Parte 2:** Dos cajas. Caja 1 (5R, 3A, 2V) y Caja 2 (2R, 5A, 3V). Se elige una caja al azar y se extraen 2 sin reemplazo. Dado que resultó 1 roja y 1 verde, estimar $P(\text{Caja 1} \mid \text{1 roja y 1 verde})$.

In [ ]:
np.random.seed(SEED)

# ── Parte 1 ─────────────────────────────────────────────
# Una caja: 5 rojas (0), 3 azules (1), 2 verdes (2)
caja = np.array([0]*5 + [1]*3 + [2]*2)

muestras_p1 = np.array([np.random.choice(caja, size=2, replace=False)
                         for _ in range(R)])

p_ambas_rojas = np.mean((muestras_p1[:, 0] == 0) & (muestras_p1[:, 1] == 0))
print(f"P(ambas rojas) = {p_ambas_rojas:.4f}")

# ── Parte 2 ─────────────────────────────────────────────
# Caja 1: 5R 3A 2V | Caja 2: 2R 5A 3V
caja1 = np.array([0]*5 + [1]*3 + [2]*2)
caja2 = np.array([0]*2 + [1]*5 + [2]*3)

casos_rv    = 0   # salió 1 roja y 1 verde
casos_c1_rv = 0   # de esos, vinieron de Caja 1

for _ in range(R):
    elegida = np.random.randint(0, 2)          # 0=Caja1, 1=Caja2
    caja_actual = caja1 if elegida == 0 else caja2
    m = np.random.choice(caja_actual, size=2, replace=False)

    if (0 in m) and (2 in m):                  # 1 roja Y 1 verde
        casos_rv += 1
        if elegida == 0:
            casos_c1_rv += 1

p_caja1_dado_rv = casos_c1_rv / casos_rv
print(f"P(Caja 1 | una roja y una verde) = {p_caja1_dado_rv:.4f}")

---
## Problema D – Cartas

**Codificación:** `0–3` = ases | `4–51` = resto de cartas

**Experimento base:** De una baraja estándar de 52 cartas se extraen 2 **sin reemplazo**.

- **a.** Estimar $P(\text{ambas ases})$.
- **b.** Sea $A$: la primera carta es un As, y $B$: la segunda carta es un As. Determinar si $A$ y $B$ son independientes comparando $P(A \cap B)$ con $P(A) \cdot P(B)$.

In [ ]:
np.random.seed(SEED)

baraja = np.arange(52)   # 0-3: ases, 4-51: no ases

# Extraer 2 cartas sin reemplazo en R experimentos
muestras = np.array([np.random.choice(baraja, size=2, replace=False)
                     for _ in range(R)])

es_as_1 = muestras[:, 0] < 4   # Evento A: primera carta es As
es_as_2 = muestras[:, 1] < 4   # Evento B: segunda carta es As

# a. P(ambas ases)
p_ambas_ases = np.mean(es_as_1 & es_as_2)

# b. Independencia: ¿P(A∩B) ≈ P(A)·P(B)?
p_A           = np.mean(es_as_1)
p_B           = np.mean(es_as_2)
p_A_por_B     = p_A * p_B
TOLERANCIA    = 0.0005
son_independientes = abs(p_ambas_ases - p_A_por_B) < TOLERANCIA

print(f"P(ambas ases) = {p_ambas_ases:.4f}")
print(f"P(A) * P(B) = {p_A_por_B:.4f}")
print(f"Los eventos son independientes: {son_independientes}")